# E-Commerce Customer Intelligence & Sales Analytics
## Notebook 01 — Data Audit

**IBM SkillsBuild Data Analytics with AI Internship 2026**

---

## 1. Business Objective

Analyze historical e-commerce transaction data to understand:
- Sales performance
- Customer behavior and retention
- Product performance and relationships
- Customer value and inactivity risk

The ultimate goal is to derive **evidence-based business recommendations** grounded entirely in the data.

> **Scope of this notebook:** Data audit only. No cleaning, modelling, or recommendations are made here.

## 2. Dataset Overview

| Item | Detail |
|------|--------|
| File | `../data/online_retail_II.xlsx` |
| Format | Excel workbook (.xlsx) |
| Sheets | Two sheets representing two time periods |
| Original file | **Read-only — never modified** |

Expected columns (schema will be verified against actual data):

| Column | Expected type | Description |
|--------|--------------|-------------|
| Invoice | object | Invoice number; prefixed 'C' for cancellations |
| StockCode | object | Product code |
| Description | object | Product name |
| Quantity | int/float | Units per transaction |
| InvoiceDate | datetime | Date and time of transaction |
| Price | float | Unit price (GBP) |
| Customer ID | float/object | Customer identifier |
| Country | object | Customer country |

## 3. Imports

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── Display settings ────────────────────────────────────────────────────────
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 60)
pd.set_option('display.float_format', '{:,.4f}'.format)
pd.set_option('display.width', 120)

# Suppress openpyxl data-validation warnings (cosmetic only; does not hide errors)
warnings.filterwarnings(
    'ignore',
    message='.*data validation.*',
    category=UserWarning,
    module='openpyxl'
)

# ── Paths ───────────────────────────────────────────────────────────────────
DATA_PATH    = '../data/online_retail_II.xlsx'
FIGURES_DIR  = '../outputs/figures/'
os.makedirs(FIGURES_DIR, exist_ok=True)

# ── Plot style ───────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120, 'figure.facecolor': 'white'})

print('Imports OK')
print(f'pandas  {pd.__version__}')
print(f'numpy   {np.__version__}')

## 4. Helper Functions

In [ ]:
def save_figure(filename: str) -> None:
    """Save the current matplotlib figure to the figures directory."""
    path = os.path.join(FIGURES_DIR, filename)
    plt.savefig(path, bbox_inches='tight')
    print(f'Figure saved → {path}')


def missing_summary(df: pd.DataFrame) -> pd.DataFrame:
    """Return a per-column missing-value summary for a DataFrame."""
    total = len(df)
    missing = df.isnull().sum()
    pct = (missing / total * 100).round(2)
    dtype = df.dtypes
    unique = df.nunique()
    summary = pd.DataFrame({
        'dtype': dtype,
        'missing_count': missing,
        'missing_pct': pct,
        'unique_values': unique,
    })
    return summary


def sheet_profile(df: pd.DataFrame, sheet_name: str) -> None:
    """Print a top-level profile for a single sheet's DataFrame."""
    print(f'\n{"=" * 60}')
    print(f'SHEET: {sheet_name}')
    print(f'{"=" * 60}')
    print(f'  Rows                : {len(df):,}')
    print(f'  Columns             : {df.shape[1]}')
    print(f'  Column names        : {list(df.columns)}')
    print(f'  Memory usage        : {df.memory_usage(deep=True).sum() / 1e6:.2f} MB')

    # Key business counts
    if 'Invoice' in df.columns:
        print(f'  Unique invoices     : {df["Invoice"].nunique():,}')
    if 'StockCode' in df.columns:
        print(f'  Unique products     : {df["StockCode"].nunique():,}')
    if 'Customer ID' in df.columns:
        print(f'  Unique customers    : {df["Customer ID"].nunique():,}')
    if 'Country' in df.columns:
        print(f'  Unique countries    : {df["Country"].nunique():,}')

    # Date range
    if 'InvoiceDate' in df.columns:
        print(f'  Min InvoiceDate     : {df["InvoiceDate"].min()}')
        print(f'  Max InvoiceDate     : {df["InvoiceDate"].max()}')

    # Quantity anomalies
    if 'Quantity' in df.columns:
        print(f'  Negative Quantity   : {(df["Quantity"] < 0).sum():,}')
        print(f'  Zero Quantity       : {(df["Quantity"] == 0).sum():,}')

    # Price anomalies
    if 'Price' in df.columns:
        print(f'  Negative Price      : {(df["Price"] < 0).sum():,}')
        print(f'  Zero Price          : {(df["Price"] == 0).sum():,}')

    # Cancellations (invoices starting with 'C')
    if 'Invoice' in df.columns:
        cancel_mask = df['Invoice'].astype(str).str.startswith('C')
        print(f'  Cancellation rows   : {cancel_mask.sum():,}')

    # Duplicates
    print(f'  Duplicate rows      : {df.duplicated().sum():,}')


def quality_table(df: pd.DataFrame) -> pd.DataFrame:
    """
    Build a data-quality summary table.
    Includes dtype, missing count/%, unique values, and min/max where meaningful.
    """
    rows = []
    for col in df.columns:
        series = df[col]
        n_missing = series.isnull().sum()
        pct_missing = round(n_missing / len(df) * 100, 2)
        n_unique = series.nunique()

        # Min / max only for numeric and datetime columns
        if pd.api.types.is_numeric_dtype(series) or pd.api.types.is_datetime64_any_dtype(series):
            col_min = series.min()
            col_max = series.max()
        else:
            col_min = '-'
            col_max = '-'

        rows.append({
            'Column': col,
            'Data Type': str(series.dtype),
            'Missing Count': n_missing,
            'Missing %': pct_missing,
            'Unique Values': n_unique,
            'Min': col_min,
            'Max': col_max,
        })
    return pd.DataFrame(rows).set_index('Column')


print('Helper functions defined.')

## 5. Dataset Loading

Load every sheet separately without concatenating. Schema is inspected from the actual file — nothing is assumed.

In [ ]:
# Read all sheet names first — do not assume names
xl_file = pd.ExcelFile(DATA_PATH, engine='openpyxl')
sheet_names = xl_file.sheet_names
print(f'Sheets found in workbook: {sheet_names}')

In [ ]:
# Load each sheet into a dictionary keyed by sheet name
# dtype=str for Invoice and StockCode to preserve leading zeros / letter prefixes
sheets: dict[str, pd.DataFrame] = {}

for name in sheet_names:
    print(f'Loading sheet: "{name}" ...', end=' ')
    df = pd.read_excel(
        DATA_PATH,
        sheet_name=name,
        engine='openpyxl',
        dtype={'Invoice': str, 'StockCode': str},
    )
    sheets[name] = df
    print(f'{len(df):,} rows loaded.')

print('\nAll sheets loaded successfully.')

## 6. Sheet Inspection

Each sheet is profiled independently before any concatenation.

In [ ]:
for name, df in sheets.items():
    sheet_profile(df, name)

In [ ]:
# Data types per sheet
for name, df in sheets.items():
    print(f'\n--- dtypes: {name} ---')
    print(df.dtypes)

In [ ]:
# First 5 rows of each sheet
for name, df in sheets.items():
    print(f'\n--- Head: {name} ---')
    display(df.head())

## 7. Schema Inspection

Verify actual column names against expectations and check for schema differences between sheets.

In [ ]:
EXPECTED_COLUMNS = ['Invoice', 'StockCode', 'Description', 'Quantity',
                    'InvoiceDate', 'Price', 'Customer ID', 'Country']

for name, df in sheets.items():
    actual = list(df.columns)
    extra   = [c for c in actual   if c not in EXPECTED_COLUMNS]
    missing = [c for c in EXPECTED_COLUMNS if c not in actual]
    print(f'\nSheet: {name}')
    print(f'  Actual columns  : {actual}')
    print(f'  Extra columns   : {extra   if extra   else "None"}')
    print(f'  Missing columns : {missing if missing else "None"}')

In [ ]:
# Check whether column sets are identical across all sheets
col_sets = [set(df.columns) for df in sheets.values()]
if all(cs == col_sets[0] for cs in col_sets):
    print('All sheets share identical column sets.')
else:
    print('WARNING: column sets differ between sheets.')
    for name, df in sheets.items():
        print(f'  {name}: {set(df.columns)}')

## 8. Data-Quality Audit

Comprehensive data-quality assessment per sheet, then on the combined dataset.

### 8.1 Quality Table — Per Sheet

In [ ]:
for name, df in sheets.items():
    print(f'\n=== Data Quality Table: {name} ===')
    display(quality_table(df))

### 8.2 Combine Sheets

After per-sheet inspection, combine into a single DataFrame for dataset-wide analysis. A `sheet` column is added for traceability.

In [ ]:
# Tag each row with its source sheet, then concatenate
tagged = []
for name, df in sheets.items():
    df_copy = df.copy()
    df_copy['_sheet'] = name
    tagged.append(df_copy)

combined = pd.concat(tagged, ignore_index=True)
print(f'Combined rows: {len(combined):,}')
print(f'Expected sum : {sum(len(df) for df in sheets.values()):,}')

# Sanity check: combined row count == sum of individual sheets
assert len(combined) == sum(len(df) for df in sheets.values()), \
    'ASSERTION FAILED: row count mismatch after concatenation'
print('Assertion passed: row counts match.')

### 8.3 InvoiceDate Parsing

In [ ]:
# Count nulls before datetime coercion
nulls_before = combined['InvoiceDate'].isnull().sum()

combined['InvoiceDate'] = pd.to_datetime(combined['InvoiceDate'], errors='coerce')

nulls_after = combined['InvoiceDate'].isnull().sum()
new_nulls   = nulls_after - nulls_before

print(f'Nulls in InvoiceDate before coercion : {nulls_before:,}')
print(f'Nulls in InvoiceDate after coercion  : {nulls_after:,}')
print(f'New nulls introduced by coercion     : {new_nulls:,}')

# Assertion: date coercion should not introduce unexpected nulls
assert new_nulls == 0, (
    f'ASSERTION FAILED: {new_nulls:,} rows have unparseable InvoiceDate values.'
)
print('Assertion passed: no unexpected date nulls introduced.')

### 8.4 Duplicate Rows

In [ ]:
dup_count = combined.duplicated().sum()
print(f'Fully duplicated rows (all columns)   : {dup_count:,}')
print(f'As percentage of total rows           : {dup_count / len(combined) * 100:.2f}%')

if dup_count > 0:
    print('\nSample of duplicated rows:')
    display(combined[combined.duplicated(keep=False)].head(10))

### 8.5 Invoice Anomalies

In [ ]:
# Unique invoice counts
total_invoices = combined['Invoice'].nunique()
print(f'Total unique invoices: {total_invoices:,}')

# Invoice format analysis
invoice_str = combined['Invoice'].astype(str)

starts_with_C = invoice_str.str.startswith('C')
numeric_only  = invoice_str.str.match(r'^\d+$')
C_prefix      = invoice_str.str.match(r'^C\d+$')
other_format  = ~(numeric_only | C_prefix)

print(f'\nInvoice format breakdown:')
print(f'  Numeric only (e.g. 489434)         : {numeric_only.sum():,}')
print(f'  C-prefixed   (e.g. C489434)         : {C_prefix.sum():,}')
print(f'  Other format                        : {other_format.sum():,}')

if other_format.sum() > 0:
    print('\nSample of non-standard invoice values:')
    display(combined.loc[other_format, 'Invoice'].value_counts().head(20))

### 8.6 StockCode Anomalies

In [ ]:
# Standard StockCode: 5-digit numeric optionally followed by one letter
standard_stock = combined['StockCode'].astype(str).str.match(r'^\d{5}[A-Za-z]?$')
non_standard   = ~standard_stock

print(f'Standard StockCode patterns   : {standard_stock.sum():,}')
print(f'Non-standard StockCode values : {non_standard.sum():,}')

if non_standard.sum() > 0:
    print('\nNon-standard StockCode value counts (top 30):')
    display(combined.loc[non_standard, 'StockCode'].value_counts().head(30))

### 8.7 Missing Customer ID

In [ ]:
total_rows        = len(combined)
rows_with_cid     = combined['Customer ID'].notna().sum()
rows_without_cid  = combined['Customer ID'].isna().sum()
pct_missing_cid   = rows_without_cid / total_rows * 100
unique_customers  = combined['Customer ID'].nunique()

print(f'Total rows                   : {total_rows:,}')
print(f'Rows with Customer ID        : {rows_with_cid:,}')
print(f'Rows without Customer ID     : {rows_without_cid:,}')
print(f'Missing Customer ID (%)      : {pct_missing_cid:.2f}%')
print(f'Unique identified customers  : {unique_customers:,}')

### 8.8 Missing Description

In [ ]:
missing_desc = combined['Description'].isna().sum()
print(f'Rows with missing Description : {missing_desc:,} '
      f'({missing_desc / total_rows * 100:.2f}%)')

# Products appearing under multiple descriptions
desc_per_product = (
    combined.dropna(subset=['Description'])
    .groupby('StockCode')['Description']
    .nunique()
)
multi_desc = desc_per_product[desc_per_product > 1]
print(f'\nProducts with > 1 unique description : {len(multi_desc):,}')
if len(multi_desc) > 0:
    print('Top 10 products with most description variants:')
    display(multi_desc.sort_values(ascending=False).head(10))

## 9. Transaction Classification Audit (Cancellations)

Investigate how cancellations are represented. Do not assume — verify from the data.

In [ ]:
# Identify cancellation rows by Invoice prefix
cancel_mask    = combined['Invoice'].astype(str).str.startswith('C')
cancel_df      = combined[cancel_mask].copy()
non_cancel_df  = combined[~cancel_mask].copy()

n_cancel_rows  = cancel_mask.sum()
n_cancel_inv   = cancel_df['Invoice'].nunique()
total_inv      = combined['Invoice'].nunique()
pct_cancel_inv = n_cancel_inv / total_inv * 100

print(f'Cancellation rows            : {n_cancel_rows:,}')
print(f'Cancellation invoices        : {n_cancel_inv:,}')
print(f'Total invoices               : {total_inv:,}')
print(f'Cancellation % of invoices   : {pct_cancel_inv:.2f}%')

In [ ]:
# Are cancellation invoices always associated with negative quantities?
cancel_qty_stats = cancel_df['Quantity'].describe()
print('Quantity distribution on cancellation invoices:')
print(cancel_qty_stats)

cancel_positive_qty = (cancel_df['Quantity'] > 0).sum()
cancel_zero_qty     = (cancel_df['Quantity'] == 0).sum()
cancel_negative_qty = (cancel_df['Quantity'] < 0).sum()

print(f'\nCancellation rows with positive Quantity : {cancel_positive_qty:,}')
print(f'Cancellation rows with zero Quantity     : {cancel_zero_qty:,}')
print(f'Cancellation rows with negative Quantity : {cancel_negative_qty:,}')

In [ ]:
# Are there negative quantities outside cancellation invoices?
neg_qty_non_cancel = ((~cancel_mask) & (combined['Quantity'] < 0)).sum()
print(f'Negative Quantity rows on NON-cancellation invoices: {neg_qty_non_cancel:,}')

if neg_qty_non_cancel > 0:
    print('\nSample of non-cancellation rows with negative Quantity:')
    display(
        combined[(~cancel_mask) & (combined['Quantity'] < 0)]
        [['Invoice', 'StockCode', 'Description', 'Quantity', 'Price', 'Customer ID']]
        .head(10)
    )

In [ ]:
# Cancellation revenue summary
cancel_df['_revenue'] = cancel_df['Quantity'] * cancel_df['Price']
cancel_qty_total   = cancel_df['Quantity'].sum()
cancel_value_total = cancel_df['_revenue'].sum()

print(f'Total Quantity on cancellation rows   : {cancel_qty_total:,.0f}')
print(f'Total Revenue on cancellation rows    : £{cancel_value_total:,.2f}')

## 10. Revenue Audit

Calculate a **temporary** transaction-level revenue field (`Quantity × Price`). The original data is not modified.

In [ ]:
# Compute revenue on a copy — original 'combined' columns untouched
combined['_revenue'] = combined['Quantity'] * combined['Price']

# Verify the calculation on a sample
sample = combined[['Quantity', 'Price', '_revenue']].head(5)
manual_check = (sample['Quantity'] * sample['Price']).round(6)
assert (manual_check == sample['_revenue'].round(6)).all(), \
    'ASSERTION FAILED: Revenue != Quantity * Price'
print('Assertion passed: _revenue == Quantity × Price')
display(sample)

In [ ]:
pos_mask  = combined['_revenue'] > 0
neg_mask  = combined['_revenue'] < 0
zero_mask = combined['_revenue'] == 0

n_pos  = pos_mask.sum()
n_neg  = neg_mask.sum()
n_zero = zero_mask.sum()

total_gross    = combined['_revenue'].sum()
total_positive = combined.loc[pos_mask,  '_revenue'].sum()
total_negative = combined.loc[neg_mask,  '_revenue'].sum()

print('Revenue Audit')
print(f'  Positive transactions  : {n_pos:,}  |  Total positive revenue : £{total_positive:,.2f}')
print(f'  Negative transactions  : {n_neg:,}  |  Total negative revenue : £{total_negative:,.2f}')
print(f'  Zero transactions      : {n_zero:,}')
print(f'  Net gross transaction value (all rows)  : £{total_gross:,.2f}')
print()
print('Interpretation:')
print('  Positive revenue = normal sales transactions.')
print('  Negative revenue = returns/cancellations (Quantity < 0).')
print('  Zero revenue     = rows where Quantity=0 OR Price=0 (no economic value exchanged).')
print('  Net gross value  = positive + negative (i.e. revenue after returns).')

## 11. Date Audit

In [ ]:
date_series = combined['InvoiceDate'].dropna()

earliest       = date_series.min()
latest         = date_series.max()
n_unique_dates = date_series.dt.date.nunique()
n_unique_months = date_series.dt.to_period('M').nunique()
n_unique_years  = date_series.dt.year.nunique()

print(f'Earliest transaction  : {earliest}')
print(f'Latest transaction    : {latest}')
print(f'Unique calendar dates : {n_unique_dates:,}')
print(f'Unique months         : {n_unique_months}')
print(f'Unique years          : {n_unique_years}')

In [ ]:
# Transactions per year
print('\nTransactions per year:')
display(combined['InvoiceDate'].dt.year.value_counts().sort_index())

# Transactions per month (YYYY-MM)
print('\nTransactions per month:')
monthly = (
    combined['InvoiceDate']
    .dt.to_period('M')
    .value_counts()
    .sort_index()
)
display(monthly)

In [ ]:
# Check for unexpected/future dates
future_dates = combined[combined['InvoiceDate'] > pd.Timestamp.now()]
print(f'Rows with future InvoiceDate : {len(future_dates):,}')

# Check for suspiciously old dates (before 2000)
old_dates = combined[combined['InvoiceDate'] < pd.Timestamp('2000-01-01')]
print(f'Rows with InvoiceDate < 2000 : {len(old_dates):,}')

## 12. Customer Data Audit

In [ ]:
identified = combined[combined['Customer ID'].notna()].copy()

# Transactions per customer
txn_per_customer = identified.groupby('Customer ID').size()
print('Transactions per customer (distribution):')
print(txn_per_customer.describe(percentiles=[0.25, 0.5, 0.75, 0.90, 0.95, 0.99]))

In [ ]:
# Revenue per customer (positive transactions only, identified customers)
identified['_revenue'] = identified['Quantity'] * identified['Price']
rev_per_customer = identified[identified['_revenue'] > 0].groupby('Customer ID')['_revenue'].sum()

print('Positive revenue per customer (distribution):')
print(rev_per_customer.describe(percentiles=[0.25, 0.5, 0.75, 0.90, 0.95, 0.99]))

## 13. Product Audit

In [ ]:
unique_stockcodes = combined['StockCode'].nunique()
unique_descs      = combined['Description'].nunique()
missing_descs     = combined['Description'].isna().sum()

print(f'Unique StockCodes            : {unique_stockcodes:,}')
print(f'Unique Descriptions          : {unique_descs:,}')
print(f'Missing Descriptions         : {missing_descs:,}')

In [ ]:
# Products with negative quantities
neg_qty_products = (
    combined[combined['Quantity'] < 0]
    .groupby('StockCode')['Quantity']
    .sum()
    .sort_values()
)
print(f'Products with negative total Quantity : {len(neg_qty_products):,}')
print('\nTop 10 by most negative quantity:')
display(neg_qty_products.head(10))

In [ ]:
# Products with zero or negative prices
zero_price_products = combined[combined['Price'] <= 0]['StockCode'].value_counts()
print(f'Products with Price <= 0 : {len(zero_price_products):,}')
if len(zero_price_products) > 0:
    display(zero_price_products.head(20))

In [ ]:
# Top 10 products by transaction row count
top_products = (
    combined
    .groupby(['StockCode', 'Description'])
    .size()
    .reset_index(name='transaction_count')
    .sort_values('transaction_count', ascending=False)
    .head(10)
    .reset_index(drop=True)
)
print('Top 10 products by transaction row count:')
display(top_products)

## 14. Initial Visualizations

Five targeted exploratory charts. No analysis conclusions are drawn here — charts are evidence gathering only.

### 14.1 Transactions by Month

In [ ]:
monthly_txn = (
    combined.dropna(subset=['InvoiceDate'])
    .assign(month=lambda d: d['InvoiceDate'].dt.to_period('M'))
    .groupby('month')
    .size()
    .sort_index()
)

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(
    range(len(monthly_txn)),
    monthly_txn.values,
    color=sns.color_palette('muted')[0],
    edgecolor='white',
    linewidth=0.5,
)
ax.set_xticks(range(len(monthly_txn)))
ax.set_xticklabels(
    [str(p) for p in monthly_txn.index],
    rotation=45, ha='right', fontsize=9
)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.set_title('Transaction Row Count by Month', fontsize=14, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Number of Transaction Rows')
plt.tight_layout()
save_figure('01_transactions_by_month.png')
plt.show()

### 14.2 Positive vs Negative Quantity Transactions

In [ ]:
qty_sign = pd.cut(
    combined['Quantity'],
    bins=[-np.inf, -0.001, 0, np.inf],
    labels=['Negative', 'Zero', 'Positive']
).value_counts().reindex(['Positive', 'Zero', 'Negative'])

colors = [sns.color_palette('muted')[2], sns.color_palette('muted')[7], sns.color_palette('muted')[3]]

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(qty_sign.index, qty_sign.values, color=colors, edgecolor='white')
for bar, val in zip(bars, qty_sign.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(qty_sign.values) * 0.01,
        f'{int(val):,}',
        ha='center', va='bottom', fontsize=10
    )
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.set_title('Transaction Rows by Quantity Sign', fontsize=14, fontweight='bold')
ax.set_xlabel('Quantity Category')
ax.set_ylabel('Number of Rows')
plt.tight_layout()
save_figure('02_quantity_sign_distribution.png')
plt.show()

### 14.3 Missing Customer ID Percentage

In [ ]:
cid_present = combined['Customer ID'].notna().sum()
cid_missing = combined['Customer ID'].isna().sum()

fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(
    [cid_present, cid_missing],
    labels=['Customer ID present', 'Customer ID missing'],
    autopct='%1.2f%%',
    startangle=90,
    colors=[sns.color_palette('muted')[0], sns.color_palette('muted')[3]],
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5},
)
ax.set_title('Missing Customer ID\n(all transaction rows)', fontsize=13, fontweight='bold')
plt.tight_layout()
save_figure('03_missing_customer_id.png')
plt.show()

### 14.4 Revenue Distribution (Valid Positive Transactions)

In [ ]:
# Valid positive: Quantity > 0, Price > 0, Customer ID present
valid_pos = combined[
    (combined['Quantity'] > 0) &
    (combined['Price'] > 0)
]['_revenue'].dropna()

# Cap at 99th percentile for readable chart
p99 = valid_pos.quantile(0.99)
clipped = valid_pos[valid_pos <= p99]

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(clipped, bins=80, color=sns.color_palette('muted')[0], edgecolor='white', linewidth=0.4)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.set_title(
    f'Revenue Distribution — Positive Transactions (capped at 99th pct = £{p99:,.2f})',
    fontsize=12, fontweight='bold'
)
ax.set_xlabel('Transaction Revenue (£)')
ax.set_ylabel('Number of Transactions')
plt.tight_layout()
save_figure('04_revenue_distribution.png')
plt.show()

print(f'Transactions plotted (≤ p99) : {len(clipped):,} of {len(valid_pos):,}')

### 14.5 Top 10 Products by Transaction Count

In [ ]:
top10 = (
    combined
    .groupby('StockCode')['Invoice']
    .count()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
    .rename(columns={'Invoice': 'transaction_count'})
)

# Add description
desc_map = (
    combined.dropna(subset=['Description'])
    .groupby('StockCode')['Description']
    .agg(lambda s: s.mode().iloc[0] if len(s.mode()) > 0 else s.iloc[0])
)
top10['description'] = top10['StockCode'].map(desc_map).str[:40]
top10['label'] = top10['StockCode'] + ' — ' + top10['description'].fillna('N/A')

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(
    top10['label'][::-1],
    top10['transaction_count'][::-1],
    color=sns.color_palette('muted', n_colors=10)[::-1],
    edgecolor='white',
)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.set_title('Top 10 Products by Transaction Row Count', fontsize=13, fontweight='bold')
ax.set_xlabel('Number of Transaction Rows')
ax.set_ylabel('Product')
plt.tight_layout()
save_figure('05_top10_products.png')
plt.show()

## 15. Validation Checks

Sanity checks to confirm the audit is internally consistent. If any assertion fails, the cause is investigated rather than suppressed.

In [ ]:
print('Running validation checks...')

# 1. Row count
expected_rows = sum(len(df) for df in sheets.values())
assert len(combined) == expected_rows, \
    f'Row count mismatch: combined={len(combined)}, expected={expected_rows}'
print(f'[PASS] Row count: {len(combined):,} == sum of sheets ({expected_rows:,})')

# 2. Original columns still present (no silent column drop)
for col in EXPECTED_COLUMNS:
    if col in combined.columns:
        assert combined[col].shape[0] == len(combined), \
            f'Column {col} has wrong row count'
print(f'[PASS] All expected columns present and complete.')

# 3. Revenue calculation reproducibility
recalc = combined['Quantity'] * combined['Price']
assert (recalc.round(8) == combined['_revenue'].round(8)).all(), \
    'Revenue recalculation mismatch'
print(f'[PASS] Revenue reproducible: Quantity × Price == _revenue on all rows.')

# 4. Cancellation count reproducibility
c_count_a = combined['Invoice'].astype(str).str.startswith('C').sum()
c_count_b = combined['Invoice'].astype(str).str.startswith('C').sum()
assert c_count_a == c_count_b, 'Cancellation count not reproducible'
print(f'[PASS] Cancellation count reproducible: {c_count_a:,} rows.')

# 5. Date nulls after coercion
date_nulls = combined['InvoiceDate'].isnull().sum()
if date_nulls > 0:
    print(f'[INFO] {date_nulls:,} InvoiceDate nulls present — investigate before cleaning.')
else:
    print('[PASS] No InvoiceDate nulls after coercion.')

print('\nAll validation checks complete.')

## 16. Initial Data Audit Findings

All findings below are calculated directly from the dataset. No values are assumed or fabricated.

> **Note:** Execute all cells above before reading this section. The numbers reported here are computed dynamically.

In [ ]:
# ── Re-derive all finding values from the live data ─────────────────────────

f_total_rows       = len(combined)
f_sheet_names      = list(sheets.keys())
f_sheet_rows       = {n: len(d) for n, d in sheets.items()}
f_date_min         = combined['InvoiceDate'].min()
f_date_max         = combined['InvoiceDate'].max()
f_n_months         = combined['InvoiceDate'].dt.to_period('M').nunique()
f_dup_rows         = combined.duplicated().sum()
f_dup_pct          = f_dup_rows / f_total_rows * 100
f_cid_missing      = combined['Customer ID'].isna().sum()
f_cid_missing_pct  = f_cid_missing / f_total_rows * 100
f_unique_customers = combined['Customer ID'].nunique()
f_cancel_rows      = combined['Invoice'].astype(str).str.startswith('C').sum()
f_cancel_inv       = combined.loc[combined['Invoice'].astype(str).str.startswith('C'), 'Invoice'].nunique()
f_total_inv        = combined['Invoice'].nunique()
f_cancel_pct_inv   = f_cancel_inv / f_total_inv * 100
f_neg_qty_non_c    = ((~combined['Invoice'].astype(str).str.startswith('C')) & (combined['Quantity'] < 0)).sum()
f_pos_rev          = combined.loc[combined['_revenue'] > 0, '_revenue'].sum()
f_neg_rev          = combined.loc[combined['_revenue'] < 0, '_revenue'].sum()
f_net_rev          = combined['_revenue'].sum()
f_zero_price       = (combined['Price'] == 0).sum()
f_non_std_stock    = (~combined['StockCode'].astype(str).str.match(r'^\d{5}[A-Za-z]?$')).sum()
f_missing_desc     = combined['Description'].isna().sum()
f_multi_desc       = (desc_per_product[desc_per_product > 1]).shape[0]

print('Finding values computed.')

In [ ]:
findings = [
    {
        'id': 1,
        'finding': f'The dataset spans {f_n_months} calendar months from '
                   f'{f_date_min.strftime("%Y-%m-%d")} to '
                   f'{f_date_max.strftime("%Y-%m-%d")}, '
                   f'loaded from {len(f_sheet_names)} sheets '
                   f'({" and ".join(f"{n}: {r:,} rows" for n, r in f_sheet_rows.items())}) '
                   f'totalling {f_total_rows:,} transaction rows.',
        'evidence': f'pd.concat of {f_sheet_names}; row count assertion passed.',
        'business_relevance': 'Determines the temporal scope of all trend, cohort, and seasonality analysis.'
    },
    {
        'id': 2,
        'finding': f'Customer ID is missing for {f_cid_missing:,} rows '
                   f'({f_cid_missing_pct:.2f}% of all transaction rows). '
                   f'{f_unique_customers:,} unique customers are identified.',
        'evidence': 'combined["Customer ID"].isna().sum()',
        'business_relevance': 'Customer-level analyses (RFM, segmentation, retention, CLV) can only be performed '
                              f'on the {100 - f_cid_missing_pct:.2f}% of rows with a Customer ID. '
                              'Transaction-level revenue analysis can use the full dataset.'
    },
    {
        'id': 3,
        'finding': f'{f_dup_rows:,} fully duplicated rows were found ({f_dup_pct:.2f}% of all rows).',
        'evidence': 'combined.duplicated().sum()',
        'business_relevance': 'Duplicated rows will inflate transaction counts and revenue figures unless removed during cleaning.'
    },
    {
        'id': 4,
        'finding': f'Cancellation invoices (Invoice starting with "C") account for '
                   f'{f_cancel_inv:,} unique invoices ({f_cancel_pct_inv:.2f}% of all invoices) '
                   f'across {f_cancel_rows:,} transaction rows.',
        'evidence': 'Invoice.str.startswith("C")',
        'business_relevance': 'Cancellations represent a measurable return/refund rate. '
                              'Net revenue calculations must exclude or offset cancellation rows.'
    },
    {
        'id': 5,
        'finding': f'{f_neg_qty_non_c:,} rows have negative Quantity on non-cancellation invoices.',
        'evidence': '(~Invoice.str.startswith("C")) & (Quantity < 0)',
        'business_relevance': 'These rows require investigation during cleaning — they are anomalous '
                              'and cannot be treated as normal sales or standard cancellations.'
    },
    {
        'id': 6,
        'finding': f'Total gross positive transaction value: £{f_pos_rev:,.2f}. '
                   f'Total negative transaction value (returns): £{f_neg_rev:,.2f}. '
                   f'Net value (gross − returns): £{f_net_rev:,.2f}.',
        'evidence': 'Revenue = Quantity × Price, split by sign.',
        'business_relevance': 'The difference between gross and net revenue quantifies the economic '
                              'impact of returns and must be tracked as a KPI.'
    },
    {
        'id': 7,
        'finding': f'{f_zero_price:,} transaction rows have a Price of exactly zero.',
        'evidence': '(combined["Price"] == 0).sum()',
        'business_relevance': 'Zero-price rows contribute no revenue. They may represent gifts, samples, '
                              'data-entry errors, or internal transfers. A decision on how to handle them is needed before modelling.'
    },
    {
        'id': 8,
        'finding': f'{f_non_std_stock:,} rows contain non-standard StockCode values '
                   f'(do not match the 5-digit pattern).',
        'evidence': 'StockCode.str.match(r"^\\d{5}[A-Za-z]?$") negated.',
        'business_relevance': 'Non-standard codes may represent postage, bank charges, manual adjustments, '
                              'or test entries. They should be catalogued and likely excluded from product-level analysis.'
    },
    {
        'id': 9,
        'finding': f'{f_missing_desc:,} rows have a missing Description.',
        'evidence': 'combined["Description"].isna().sum()',
        'business_relevance': 'Missing descriptions limit product-name-based analysis and labelling in dashboards.'
    },
    {
        'id': 10,
        'finding': f'{f_multi_desc:,} StockCodes appear under more than one distinct Description.',
        'evidence': 'groupby("StockCode")["Description"].nunique() > 1',
        'business_relevance': 'Description inconsistency means StockCode, not Description, must be the '
                              'canonical product identifier in all downstream analyses.'
    },
]

for f in findings:
    print(f'Finding {f["id"]}:')
    print(f'  Finding           : {f["finding"]}')
    print(f'  Evidence          : {f["evidence"]}')
    print(f'  Business relevance: {f["business_relevance"]}')
    print()

## 17. Next-Step Recommendations

These recommendations follow directly from the findings above. **No cleaning, modelling, or analysis is performed in this notebook.**

| # | Issue identified | Recommended action in Notebook 02 |
|---|-----------------|-----------------------------------|
| 1 | Duplicated rows | Remove fully duplicated rows after confirming they are not legitimate re-entries |
| 2 | Missing Customer ID | Retain rows for revenue analysis; flag and set aside for customer-level analysis |
| 3 | Cancellation invoices | Separate into a cancellation ledger; do not remove — use for net revenue calculation |
| 4 | Negative Qty on non-cancellation invoices | Investigate each case; decide inclusion rule |
| 5 | Zero-price rows | Classify by StockCode pattern; decide whether to include in revenue KPIs |
| 6 | Non-standard StockCodes | Build an exclusion list (postage, test, adjustments) |
| 7 | Multiple descriptions per StockCode | Standardise to modal description per StockCode |
| 8 | InvoiceDate parsing | Already resolved — confirm no new nulls in combined dataset |

---

*End of Notebook 01 — Data Audit. Do not proceed to cleaning or analysis without resolving the items above.*